# Zero-Shot Chain-of-Thought Prompting

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand the difference between standard zero-shot and zero-shot CoT prompting
2. Send structured chat-completion requests to a HuggingFace-hosted model
3. Observe how adding *'Let's think step by step'* changes model reasoning
4. Compare zero-shot CoT against few-shot CoT on the same problem


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install requests
```


In [ ]:
# pip install requests  # run once in terminal
import os
import requests


---

## 2. API Configuration

Store your HuggingFace token in the `HUGGINGFACE_API_KEY` environment variable:  
`export HUGGINGFACE_API_KEY=hf_...`  (Linux/macOS) or  
`$env:HUGGINGFACE_API_KEY='hf_...'`  (PowerShell)


In [ ]:
API_URL = "https://router.huggingface.co/novita/v3/openai/chat/completions"

# Load the token from an environment variable — never hard-code secrets in notebooks
HF_TOKEN = os.environ.get("HUGGINGFACE_API_KEY", "your-huggingface-api-key-here")
headers = {"Authorization": f"Bearer {HF_TOKEN}"}


---

## 3. Helper Function


In [ ]:
def send_request(messages):
    """Send a chat-completion request and return the model's reply text."""
    payload = {
        "messages": messages,
        "model": "deepseek/deepseek-v3-0324",
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    try:
        return response.json()["choices"][0]["message"]["content"]
    except KeyError:
        return f"Error in response: {response.text}"


---

## 4. Zero-Shot Chain-of-Thought

The only change from a standard zero-shot prompt is the trigger phrase  
**'Pensemos paso a paso.'** ('Let's think step by step.'), which encourages  
the model to produce intermediate reasoning before stating the answer.


In [ ]:
zero_shot_cot_messages = [
    {
        "role": "user",
        "content": (
            "Si el triple de un número sumado a 5 es igual a 20, ¿cuál es el número? "
            "Pensemos paso a paso."
        ),
    }
]

print("=== Zero-shot Chain-of-Thought ===")
print(send_request(zero_shot_cot_messages))


> **Output interpretation:** The model should enumerate algebraic steps before reaching the answer (x = 5). If it jumps straight to the answer without working, the CoT trigger failed — try a larger model or re-phrase the trigger to 'Let us solve this step by step'.


---

## 5. Few-Shot Chain-of-Thought

Providing a worked example (the '2x + 4 = 14' problem) before the target question  
gives the model a concrete reasoning template to follow.


In [ ]:
few_shot_cot_messages = [
    {
        "role": "user",
        "content": (
            "Ejemplo de resolución:\n"
            "Para el problema 'Si el doble de un número sumado a 4 es igual a 14, ¿cuál es el número?':\n"
            "Planteamos la ecuación: 2x + 4 = 14.\n"
            "Restamos 4 a ambos lados: 2x = 10.\n"
            "Dividimos entre 2: x = 5.\n\n"
            "Ahora, resuelve el siguiente problema usando el mismo método:\n"
            "Si el triple de un número sumado a 5 es igual a 20, ¿cuál es el número?"
        ),
    }
]

print("=== Few-shot Chain-of-Thought ===")
print(send_request(few_shot_cot_messages))


> **Output interpretation:** With the worked example, the model mirrors the step-by-step format of the exemplar. Compare the structure and verbosity with the zero-shot CoT output above — few-shot CoT typically produces more uniform formatting because the model has a template to follow.


---

## Summary

| Technique | Trigger | Notes |
|---|---|---|
| Zero-shot CoT | 'Let's think step by step' | No examples needed; effective on arithmetic/logic |
| Few-shot CoT | Worked example + question | More control over output format; costs extra tokens |
